## Implementing a transformer from scratch on the Shakespear dataset

In [29]:
import numpy as np 
import matplotlib.pyplot as plt 
import os, pickle, sys, torch, random 
import torch.nn as nn 
from torch.nn import functional as F 
import pandas as pd 
from gensim.utils import tokenize 

In [2]:
results_path = os.path.join('results', '9_transformer_NLP')

if not os.path.exists(results_path):
    os.makedirs(results_path)

In [3]:
## exploring the dataset format
with open('data/SHAKESPEAR/Shakespear.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [4]:
print(len(text))

1115393


In [5]:
print(text[:100]) 

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [6]:
# tokenized_text = list(tokenize(text))

In [7]:
# token_list = sorted(list(set(tokenized_text)))
token_list = sorted(list(set(text)))

In [8]:
vocab_size = len(token_list)
vocab_size 

65

In [9]:
random.sample(token_list, 20) 

['i',
 'P',
 'S',
 'j',
 'w',
 'e',
 'p',
 'u',
 'G',
 'Q',
 'F',
 ' ',
 ':',
 'W',
 'y',
 'H',
 'M',
 'l',
 'n',
 'U']

In [11]:
# create a mapping from tokens to integers
stoi = {token: i for i, token in enumerate(token_list)}
itos = {i: token for i, token in enumerate(token_list)} 

encode = lambda s: [stoi[t] for t in s]            # encoder, take a string, output a list of integers 
decode = lambda l: ''.join([itos[i] for i in l])  # decoder, take a list of integers, output a string 

print(encode('catched rabbit'))
print(decode(encode('catched rabbit'))) 

[41, 39, 58, 41, 46, 43, 42, 1, 56, 39, 40, 40, 47, 58]
catched rabbit


In [12]:
random_seed = 0 
torch.manual_seed(random_seed) 

In [13]:
batch_size = 4
block_size = 8 

In [14]:
data = torch.tensor(encode(text), dtype=torch.long) 
print(data.shape, data.dtype) 
print(data[:100])

torch.Size([1115393]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [15]:
# split the data into train and validation sets
n = int(0.9 * len(data))
train_data = data[:n] 
test_data = data[n:] 

In [16]:
print(len(train_data))
print(len(test_data)) 

1003853
111540


In [17]:
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [20]:
x = train_data[:block_size]
y = train_data[1:block_size+1] 

for t in np.arange(block_size):
    context = x[:t+1]
    target = y[t] 
    print(f'when the input is {context}, output is {target}') 

when the input is tensor([18]), output is 47
when the input is tensor([18, 47]), output is 56
when the input is tensor([18, 47, 56]), output is 57
when the input is tensor([18, 47, 56, 57]), output is 58
when the input is tensor([18, 47, 56, 57, 58]), output is 1
when the input is tensor([18, 47, 56, 57, 58,  1]), output is 15
when the input is tensor([18, 47, 56, 57, 58,  1, 15]), output is 47
when the input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), output is 58


In [26]:
def get_batch(split):
    # generate a small batch of data of input x and target y 
    data = train_data if split == 'train' else test_data
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i:i+block_size] for i in ix]) 
    y = torch.stack([data[i+1:i+1+block_size] for i in ix])

    return x, y 

In [28]:
xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape) 
print(yb) 

print('---------')

for b in np.arange(batch_size):
    for t in np.arange(block_size):
        context = xb[b, :t+1]
        target = yb[b, t] 
        print(f'when the input is {context}, target is {target}')        

inputs:
torch.Size([4, 8])
tensor([[ 1, 39, 45, 43,  6,  0, 37, 53],
        [58, 46, 43,  1, 61, 43, 39, 50],
        [46, 47, 52,  1, 53, 44,  1, 57],
        [43, 44,  8,  0,  0, 29, 33, 17]])
targets:
torch.Size([4, 8])
tensor([[39, 45, 43,  6,  0, 37, 53, 59],
        [46, 43,  1, 61, 43, 39, 50,  1],
        [47, 52,  1, 53, 44,  1, 57, 59],
        [44,  8,  0,  0, 29, 33, 17, 17]])
---------
when the input is tensor([1]), target is 39
when the input is tensor([ 1, 39]), target is 45
when the input is tensor([ 1, 39, 45]), target is 43
when the input is tensor([ 1, 39, 45, 43]), target is 6
when the input is tensor([ 1, 39, 45, 43,  6]), target is 0
when the input is tensor([ 1, 39, 45, 43,  6,  0]), target is 37
when the input is tensor([ 1, 39, 45, 43,  6,  0, 37]), target is 53
when the input is tensor([ 1, 39, 45, 43,  6,  0, 37, 53]), target is 59
when the input is tensor([58]), target is 46
when the input is tensor([58, 46]), target is 43
when the input is tensor([58, 46, 

In [30]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.token_embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensor of integers
        logits = self.token_embedding_table(idx)  # (B, T, C) 

        if targets is None:
            loss = None 
        else: 
            B, T, C = logits.shape 
            logits = logits.view(B*T, C) 
            targets = targets.view(B*T) 
            loss = F.cross_entropy(logits, targets) 

        return logits, loss 
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context 
        for _ in np.arange(max_new_tokens):
            logits, loss = self.forward(idx) 
            logits = logits[:, -1, :]   # (B, C)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1) 
            # append sampled index to the running sequence 
            idx = torch.cat((idx, idx_next), dim=1) 
        return idx 

In [31]:
m = BigramLanguageModel(vocab_size=vocab_size, embedding_dim=vocab_size) 
print(np.array(m.token_embedding_table)) 

Embedding(65, 65)


In [32]:
logits, loss = m(xb, yb) 
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.8796, grad_fn=<NllLossBackward0>)


In [33]:
idx = m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)
print(idx)
print(decode(idx[0].tolist())) 

tensor([[ 0, 55, 25, 19, 43, 23, 44, 23, 40, 18, 52, 48, 21, 54,  4, 47, 29, 60,
         58,  5,  9, 19, 17, 53, 18, 63, 17, 40, 23,  6, 27, 33, 10, 60, 11, 51,
         60,  6, 29, 64, 27,  7, 64,  9, 52, 13, 52, 35, 31, 49, 44, 63, 48, 34,
          0, 53, 55, 35, 33, 50, 59, 25, 56, 44, 31, 52, 14, 55, 36, 22,  7,  4,
         64, 56, 47, 13,  3, 59, 18, 49,  8, 17, 32, 61, 32, 50, 47, 39,  1, 15,
         57, 24,  5, 52, 62, 35,  2, 22, 44, 23, 55]])

qMGeKfKbFnjIp&iQvt'3GEoFyEbK,OU:v;mv,QzO-z3nAnWSkfyjV
oqWUluMrfSnBqXJ-&zriA$uFk.ETwTlia CsL'nxW!JfKq


### The mathematical details in self-attention

In [34]:
# an example
B, T, C = 4, 8, 2  # batch, time, channels
x = torch.randn(B, T, C)
x.shape 

torch.Size([4, 8, 2])

In [35]:
# we want x[b, t] = mean_{i<=t} x[b, i]
# version 1
xbow = torch.zeros((B, T, C))
for b in np.arange(B):
    for t in np.arange(T):
        xprev = x[b, :t+1]                       # (t, C)
        xbow[b, t] = torch.mean(xprev, axis=0)   # (C,) 

In [36]:
xbow[0]

tensor([[0.0818, 1.2940],
        [0.9325, 0.2020],
        [0.3486, 0.7548],
        [0.2863, 1.2532],
        [0.2850, 1.3532],
        [0.3350, 1.2037],
        [0.2853, 0.8572],
        [0.4653, 0.8661]])

In [42]:
# we want x[b, t] = mean_{i<=t} x[b, i]
# version 2
wei = torch.tril(torch.ones(T, T))
wei = wei / torch.sum(wei, axis=1, keepdim=True) 
xbow2 = wei @ x      # (B, T, T) @ (B, T, C) --> (B, T, C) 
xbow2[0]

tensor([[0.0818, 1.2940],
        [0.9325, 0.2020],
        [0.3486, 0.7548],
        [0.2863, 1.2532],
        [0.2850, 1.3532],
        [0.3350, 1.2037],
        [0.2853, 0.8572],
        [0.4653, 0.8661]])

In [43]:
torch.allclose(xbow, xbow2)

True

In [48]:
# version 3
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x 
torch.allclose(xbow, xbow3)

True

In [56]:
# version 4: self-attention 
B, T, C = 4, 8, 32

x = torch.randn(B, T, C)

# see a single head performs self-attention 
head_size = 16 
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)                       # (B, T, 16)
q = query(x)                     # (B, T, 16)
wei = q @ k.transpose(-2, -1)    # (B, T, 16) @ (B, 16, T) --> (B, T, T)
wei = wei * head_size ** -0.5

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v 
out.shape 

torch.Size([4, 8, 16])

In [57]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4211, 0.5789, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2595, 0.3961, 0.3444, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5298, 0.1853, 0.1643, 0.1206, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1946, 0.1858, 0.2376, 0.1801, 0.2019, 0.0000, 0.0000, 0.0000],
        [0.2273, 0.0939, 0.2408, 0.2321, 0.1584, 0.0474, 0.0000, 0.0000],
        [0.1515, 0.0964, 0.1809, 0.1620, 0.1828, 0.1037, 0.1227, 0.0000],
        [0.1121, 0.1129, 0.0879, 0.0769, 0.0760, 0.1519, 0.2435, 0.1387]],
       grad_fn=<SelectBackward0>)